# 실습: ITO 고장진단 — S-파라미터 기반 1D CNN 분류

## 학습 목표
1. **데이터 구조**를 확인하고, CSV 형태의 S-파라미터(측정값)를 이해한다.
2. **전체 데이터를 한 그래프**에 그려 클래스별 파형 차이를 직관적으로 파악한다.
3. **1D CNN**으로 7개 고장/정상 클래스를 분류하는 모델을 학습한다.
4. **임의의 CSV 파일 하나**를 넣어 클래스를 판정하는 추론 과정을 실습한다.

## 데이터셋 개요
| 클래스 | 의미(강의 맥락) |
|--------|----------------|
| `Normal` | 정상 |
| `M1`, `M2`, `M3` | Minus 계열 고장 |
| `P1`, `P2`, `P3` | Plus 계열 고장 |

- 각 CSV 컬럼: `Frequency`(Hz), `Measurement`(S11 등, dB)
- 샘플당 주파수 포인트 수: **201개**

```text
ITO_dataset/
├── M1/
├── M2/
├── M3/
├── Normal/
├── P1/
├── P2/
└── P3/
```

## 실행 방법 (수강생)
1. 상단 메뉴에서 **[드라이브에 사본 저장]** 을 눌러 본인 계정으로 복사하세요.
2. 위에서부터 셀을 **순서대로** 실행하면 됩니다.
3. 데이터는 **자동으로 내려받아집니다.** Drive 업로드나 경로 입력은 필요 없습니다.
4. 학습 속도를 높이려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU** 로 설정하세요. (CPU로도 실행 가능합니다)

> 저장되는 모델·그림은 `/content/outputs` 에 생성되며, 런타임이 종료되면 사라집니다.
> 결과를 남기려면 좌측 파일 탐색기에서 내려받으세요.

## 0. 실습 데이터 자동 다운로드

강사가 공유한 `ITO_dataset.zip` 을 내려받아 압축을 풉니다.
셀을 여러 번 실행해도 이미 받아둔 데이터는 다시 내려받지 않습니다.

In [ ]:
# ============================================================
# [0] 실습 데이터 자동 다운로드
#  - Google Drive 공유 zip 을 내려받아 /content 에 풉니다.
#  - Drive 마운트나 경로 입력이 필요 없습니다.
#  - 이미 준비되어 있으면 건너뜁니다.
# ============================================================
import sys, subprocess, zipfile, shutil
from pathlib import Path

ZIP_FILE_ID = "1yuaTUaxku8zGd8oAqg1e11MotwTriCgL"   # ITO_dataset.zip 공유 파일 ID
ZIP_PATH    = Path("/content/ITO_dataset.zip")
EXTRACT_DIR = Path("/content/ITO_data")

CLASS_NAMES = ["M1", "M2", "M3", "Normal", "P1", "P2", "P3"]


def find_data_root(base: Path, max_depth: int = 3):
    """7개 클래스 폴더를 모두 가진 디렉터리를 찾는다 (없으면 None)."""
    if not base.exists():
        return None
    queue = [(base, 0)]
    while queue:
        cur, depth = queue.pop(0)
        if all((cur / c).is_dir() for c in CLASS_NAMES):
            return cur
        if depth < max_depth:
            for child in sorted(p for p in cur.iterdir() if p.is_dir()):
                if child.name in ("__MACOSX", ".ipynb_checkpoints"):
                    continue
                queue.append((child, depth + 1))
    return None


DATA_ROOT = find_data_root(EXTRACT_DIR)

if DATA_ROOT is None:
    try:
        import gdown                     # Colab에는 기본 설치되어 있습니다
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown

    if not ZIP_PATH.exists():
        print("데이터를 내려받는 중입니다. 잠시만 기다려 주세요...\n")
        gdown.download(id=ZIP_FILE_ID, output=str(ZIP_PATH), quiet=False)

    print("\n압축을 푸는 중입니다...")
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_DIR)

    shutil.rmtree(EXTRACT_DIR / "__MACOSX", ignore_errors=True)
    DATA_ROOT = find_data_root(EXTRACT_DIR)

if DATA_ROOT is None:
    raise RuntimeError(
        "데이터 준비에 실패했습니다.\n"
        " 1) 공유 링크가 '링크가 있는 모든 사용자(뷰어)'로 설정되어 있는지 확인하세요.\n"
        " 2) 동시 접속이 많으면 잠시 후 다시 실행하면 되는 경우가 많습니다.\n"
        " 3) 그래도 안 되면 아래 두 줄을 실행해 처음부터 다시 받으세요.\n"
        "    !rm -rf /content/ITO_data /content/ITO_dataset.zip"
    )

n_csv = sum(1 for _ in DATA_ROOT.rglob("*.csv"))
print("\n✅ 데이터 준비 완료")
print("DATA_ROOT   :", DATA_ROOT)
print("CSV 파일 수 :", n_csv)

데이터를 내려받는 중입니다. 잠시만 기다려 주세요...



Downloading...
From: https://drive.google.com/uc?id=1yuaTUaxku8zGd8oAqg1e11MotwTriCgL
To: /content/ITO_dataset.zip
100%|██████████| 17.4M/17.4M [00:00<00:00, 64.3MB/s]



압축을 푸는 중입니다...

✅ 데이터 준비 완료
DATA_ROOT   : /content/ITO_data/ITO_dataset
CSV 파일 수 : 5206


## 1. 라이브러리 import & 데이터 경로 설정

데이터 경로(`DATA_ROOT`)는 위 셀에서 이미 자동으로 잡혔습니다. 수정할 부분은 없습니다.

In [ ]:
# ------------------------------------------------------------
# [1] 라이브러리 & 경로 설정
# ------------------------------------------------------------
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 재현성을 위해 난수 시드 고정 (같은 조건이면 비슷한 결과가 나옴)
SEED = 42
tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# 데이터 경로: 위 [0] 셀에서 자동으로 설정된 DATA_ROOT 를 그대로 사용합니다.
# (수강생이 직접 수정할 필요가 없습니다)
# ============================================================
try:
    DATA_ROOT
except NameError:
    raise RuntimeError("먼저 맨 위의 [0] 데이터 자동 다운로드 셀을 실행하세요.")

# 학습 결과(모델, 그림)를 저장할 폴더
OUTPUT_DIR = Path("/content/outputs")

CLASS_NAMES = ["M1", "M2", "M3", "Normal", "P1", "P2", "P3"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

print("TensorFlow:", tf.__version__)
print("DATA_ROOT :", DATA_ROOT)
print("존재 여부 :", DATA_ROOT.exists())
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("하위 폴더 :", sorted([p.name for p in DATA_ROOT.iterdir() if p.is_dir()]))
print("결과 저장 :", OUTPUT_DIR)

TensorFlow: 2.20.0
DATA_ROOT : /content/ITO_data/ITO_dataset
존재 여부 : True
하위 폴더 : ['M1', 'M2', 'M3', 'Normal', 'P1', 'P2', 'P3', '_outputs']
결과 저장 : /content/outputs


## 2. 데이터 구조 확인

클래스 폴더별로 CSV가 몇 개인지, 파일 이름은 어떤 패턴인지 먼저 살펴봅니다.

> **왜 먼저 구조를 보나?**  
> AI 모델보다 중요한 것은 "데이터가 무엇을 의미하는지"입니다. 폴더=클래스, 파일=샘플이라는 규칙이 맞는지 확인해야 이후 학습이 의미가 있습니다.

In [ ]:
# ------------------------------------------------------------
# [2] 클래스별 파일 개수 / 샘플 파일명 확인
# ------------------------------------------------------------
def summarize_dataset(data_root: Path):
    """클래스 폴더를 돌며 CSV 개수를 출력합니다."""
    print(f"데이터 루트: {data_root}")
    print("-" * 45)
    total = 0
    for name in CLASS_NAMES:
        class_dir = data_root / name
        if not class_dir.is_dir():
            print(f"  {name:8s}: [폴더 없음!]")
            continue
        files = sorted(class_dir.glob("*.csv"))
        total += len(files)
        sample = files[0].name if files else "(없음)"
        print(f"  {name:8s}: {len(files):5d} 개  | 예: {sample}")
    print("-" * 45)
    print(f"  합계    : {total:5d} 개  /  {len(CLASS_NAMES)} 클래스")


summarize_dataset(DATA_ROOT)

데이터 루트: /content/ITO_data/ITO_dataset
---------------------------------------------
  M1      :   630 개  | 예: M1_S11_aug_0dB_0001.csv
  M2      :   672 개  | 예: M2_S11_aug_0dB_0001.csv
  M3      :   672 개  | 예: M3_S11_aug_0dB_0001.csv
  Normal  :  1300 개  | 예: Normal_S11_aug_0dB_0001.csv
  P1      :   630 개  | 예: P1_S11_aug_0dB_0001.csv
  P2      :   630 개  | 예: P2_S11_aug_0dB_0001.csv
  P3      :   672 개  | 예: P3_S11_aug_0dB_0001.csv
---------------------------------------------
  합계    :  5206 개  /  7 클래스


## 3. CSV 하나 열어보기

표(Frequency, Measurement)와 그래프를 함께 보면, 입력이 **이미지**가 아니라 **1차원 주파수 응답 곡선**임을 이해할 수 있습니다.

- 가로축: 주파수 (Frequency)
- 세로축: 측정값 (Measurement, 보통 S11 dB)

In [ ]:
# ------------------------------------------------------------
# [3] 단일 CSV 샘플 확인 (표 + 그래프)
# ------------------------------------------------------------
def load_csv_signal(path: Path):
    """CSV에서 Frequency, Measurement를 numpy 배열로 읽습니다."""
    df = pd.read_csv(path)
    # 컬럼 이름이 다르면 바로 에러를 내서 문제를 빨리 발견하게 함
    if "Frequency" not in df.columns or "Measurement" not in df.columns:
        raise ValueError(f"컬럼이 Frequency, Measurement 이어야 합니다: {path}")
    freq = df["Frequency"].to_numpy(dtype=np.float32)
    meas = df["Measurement"].to_numpy(dtype=np.float32)
    return freq, meas, df


# Normal 클래스의 첫 번째 파일을 예시로 사용
sample_path = sorted((DATA_ROOT / "Normal").glob("*.csv"))[0]
freq, meas, df_sample = load_csv_signal(sample_path)

print("파일:", sample_path.name)
print("포인트 수:", len(meas))
print("주파수 범위: {:.3e} ~ {:.3e} Hz".format(freq.min(), freq.max()))
display(df_sample.head(8))  # Colab/Jupyter에서 표를 예쁘게 보여줌

plt.figure(figsize=(10, 4))
plt.plot(freq / 1e9, meas, color="#d62728", linewidth=1.5)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Measurement (dB)")
plt.title(f"Single sample: {sample_path.name}")
plt.grid(True, alpha=0.3)
plt.show()

## 4. 전체 데이터셋 로드

모든 CSV를 읽어 행렬로 만듭니다.

- `X`: shape `(N, 201)` — N개 샘플, 각 샘플은 201개 Measurement
- `y`: shape `(N,)` — 정수 라벨 (0~6)
- `freq`: shape `(201,)` — 공통 주파수 축

> 파일이 수천 개면 로딩에 **1~2분** 정도 걸릴 수 있습니다. 정상입니다.

In [ ]:
# ------------------------------------------------------------
# [4] 전체 데이터셋 로드 함수
# ------------------------------------------------------------
def load_dataset(data_root: Path):
    """
    7개 클래스 폴더의 모든 CSV를 읽어 X, y로 반환합니다.

    Returns
    -------
    X : (N, L) float32
    y : (N,) int64
    freq : (L,) float32
    paths : 각 샘플의 파일 경로 리스트
    """
    X_list, y_list, paths = [], [], []
    freq_ref = None

    for class_name in CLASS_NAMES:
        class_dir = data_root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"클래스 폴더 없음: {class_dir}")

        csv_files = sorted(class_dir.glob("*.csv"))
        if not csv_files:
            raise FileNotFoundError(f"CSV 없음: {class_dir}")

        label = CLASS_TO_IDX[class_name]  # 폴더명 -> 정수 라벨
        for csv_path in csv_files:
            freq, meas, _ = load_csv_signal(csv_path)

            # 모든 샘플의 길이가 같은지 검사 (다르면 CNN 입력이 안 맞음)
            if freq_ref is None:
                freq_ref = freq
            elif len(meas) != len(freq_ref):
                raise ValueError(
                    f"길이 불일치: {csv_path} ({len(meas)} vs {len(freq_ref)})"
                )

            X_list.append(meas)
            y_list.append(label)
            paths.append(str(csv_path))

    X = np.stack(X_list, axis=0).astype(np.float32)
    y = np.asarray(y_list, dtype=np.int64)
    return X, y, freq_ref, paths


print("데이터 로딩 중... (잠시 기다려 주세요)")
X, y, freq, paths = load_dataset(DATA_ROOT)
print(f"X shape = {X.shape}  # (샘플 수, 주파수 포인트 수)")
print(f"y shape = {y.shape}")
print("클래스별 개수:", {CLASS_NAMES[i]: int(np.sum(y == i)) for i in range(len(CLASS_NAMES))})

## 5. 모든 데이터를 하나의 그래프로 시각화

- 반투명 얇은 선: **개별 샘플** (전체)
- 두꺼운 선: **클래스별 평균 파형** (대표 패턴)

> 평균 곡선이 클래스마다 어떻게 다른지 관찰해 보세요.  
> 차이가 보이면 "분류가 가능한 신호"라는 직관을 얻을 수 있습니다.

In [ ]:
# ------------------------------------------------------------
# [5] 전체 데이터 시각화 (한 그래프)
# - 샘플이 수천 개라 그리는 데 시간이 걸릴 수 있습니다.
# - 빠르게 보고 싶으면 MAX_PLOT_PER_CLASS 값을 줄이세요. (예: 50)
# ------------------------------------------------------------
COLORS = {
    "M1": "#1f77b4",
    "M2": "#ff7f0e",
    "M3": "#2ca02c",
    "Normal": "#d62728",
    "P1": "#9467bd",
    "P2": "#8c564b",
    "P3": "#e377c2",
}

# None이면 전부 그림. 정수면 클래스당 최대 그 개수만 개별 샘플로 표시
MAX_PLOT_PER_CLASS = None  # 예: 100

freq_ghz = freq / 1e9  # Hz -> GHz (축을 읽기 쉽게)

fig, ax = plt.subplots(figsize=(14, 7))

# (1) 개별 샘플: 반투명으로 겹쳐 그려 분포를 보여줌
for class_idx, class_name in enumerate(CLASS_NAMES):
    samples = X[y == class_idx]
    if MAX_PLOT_PER_CLASS is not None:
        samples = samples[:MAX_PLOT_PER_CLASS]
    color = COLORS[class_name]
    for signal in samples:
        ax.plot(freq_ghz, signal, color=color, alpha=0.03, linewidth=0.6)

# (2) 클래스 평균: 두껍게 그려 "대표 파형"을 강조
mean_handles = []
mean_labels = []
for class_idx, class_name in enumerate(CLASS_NAMES):
    mean_signal = X[y == class_idx].mean(axis=0)
    (line,) = ax.plot(
        freq_ghz,
        mean_signal,
        color=COLORS[class_name],
        linewidth=2.2,
        alpha=0.95,
    )
    mean_handles.append(line)
    mean_labels.append(f"{class_name} (n={int(np.sum(y == class_idx))})")

ax.set_xlabel("Frequency (GHz)", fontsize=12)
ax.set_ylabel("S-parameter Measurement (dB)", fontsize=12)
ax.set_title(f"ITO Fault Diagnosis — All Samples ({len(X)} files, 7 classes)", fontsize=14)
ax.grid(True, alpha=0.3)
ax.legend(mean_handles, mean_labels, loc="best", fontsize=9, framealpha=0.9)
fig.tight_layout()

save_path = OUTPUT_DIR / "all_data_visualization.png"
fig.savefig(save_path, dpi=150)
plt.show()
print("저장:", save_path)

## 6. 학습/검증/테스트 분할 & 전처리

### 왜 나누나?
- **Train**: 모델이 패턴을 배우는 데이터
- **Validation**: 학습 중 과적합(overfitting)을 감시
- **Test**: 최종 성적표 (학습에 쓰지 않은 데이터)

### StandardScaler란?
각 주파수 포인트의 평균을 0, 분산을 1에 가깝게 맞춥니다.  
딥러닝이 더 안정적으로 학습되도록 돕는 **정규화** 단계입니다.

> 중요: scaler는 **Train 통계만**으로 `fit` 합니다.  
> Test 정보를 미리 보면 "시험지를 미리 보는" 것과 같아서 공정하지 않습니다.

In [ ]:
# ------------------------------------------------------------
# [6] train / val / test 분할 + 표준화 + CNN 입력 형태로 변환
# ------------------------------------------------------------
TEST_SIZE = 0.15   # 전체의 15%를 최종 테스트로 분리
VAL_SIZE = 0.15    # 전체 기준 약 15%를 검증용으로 분리

# stratify=y : 클래스 비율을 train/test에 비슷하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=VAL_SIZE / (1.0 - TEST_SIZE),
    random_state=SEED,
    stratify=y_train,
)

print(f"Train: {X_train.shape[0]} / Val: {X_val.shape[0]} / Test: {X_test.shape[0]}")

# 표준화: fit은 train만, transform은 전부
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

# 1D CNN 입력 형태: (샘플 수, 길이, 채널 수)
# 이미지가 (H, W, C)인 것처럼, 시계열/스펙트럼은 (L, C)로 봅니다.
# 여기서 채널=1 (Measurement 한 종류)
X_train_s = X_train_s[..., np.newaxis]
X_val_s = X_val_s[..., np.newaxis]
X_test_s = X_test_s[..., np.newaxis]

print("CNN 입력 예:", X_train_s.shape, "-> (N, 201, 1)")

## 7. 1D CNN 모델 정의

### 왜 1D CNN인가?
- 이미지용 2D CNN은 "가로·세로 공간 패턴"을 봅니다.
- 우리 데이터는 **주파수 축을 따라 이어진 1차원 곡선**이므로 `Conv1D`가 자연스럽습니다.
- 합성곱 필터가 국소적인 파형 모양(피크, 딥, 기울기)을 자동으로 추출합니다.

### 층 역할 한눈에 보기
| 층 | 역할 |
|----|------|
| `Conv1D` | 지역 패턴(특징) 추출 |
| `BatchNormalization` | 학습 안정화 |
| `MaxPooling1D` | 길이를 줄이며 중요한 응답만 남김 |
| `GlobalAveragePooling1D` | 채널별 평균으로 고정 길이 벡터 생성 |
| `Dropout` | 과적합 완화 |
| `Dense(softmax)` | 7개 클래스 확률 출력 |

In [ ]:
# ------------------------------------------------------------
# [7] 1D CNN 구성
# ------------------------------------------------------------
def build_cnn(input_length: int, num_classes: int) -> keras.Model:
    """ITO S-파라미터 분류용 간단한 1D CNN."""
    inputs = keras.Input(shape=(input_length, 1), name="s_param")

    # Block 1: 넓은 필터(kernel=7)로 완만한 패턴부터 보기
    x = layers.Conv1D(32, kernel_size=7, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Block 2: 채널을 늘려 더 다양한 특징 학습
    x = layers.Conv1D(64, kernel_size=5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Block 3: 더 세밀한 패턴
    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)

    # 분류 Head
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="class_prob")(x)

    model = keras.Model(inputs, outputs, name="ITO_1D_CNN")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        # y가 정수 라벨(0~6)이므로 sparse_categorical_crossentropy 사용
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_cnn(input_length=X.shape[1], num_classes=len(CLASS_NAMES))
model.summary()

## 8. 모델 학습

- `EarlyStopping`: 검증 성능이 더 이상 좋아지지 않으면 일찍 멈춤
- `ReduceLROnPlateau`: 정체되면 학습률을 낮춰 세밀하게 조정

Colab CPU에서도 수 분 내외로 끝나는 규모입니다.

In [ ]:
# ------------------------------------------------------------
# [8] 학습
# ------------------------------------------------------------
EPOCHS = 30
BATCH_SIZE = 64

callbacks = [
    # val_accuracy가 8 epoch 동안 개선되지 않으면 중단, 최고 가중치 복원
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
    ),
    # val_loss가 정체되면 학습률을 절반으로
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-5,
    ),
]

print("학습 시작...")
history = model.fit(
    X_train_s,
    y_train,
    validation_data=(X_val_s, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

## 9. 학습 곡선 & 테스트 성능 평가

- Loss가 내려가고 Accuracy가 올라가는지 확인합니다.
- Train만 좋고 Val이 나쁘면 **과적합** 신호입니다.
- 혼동행렬(Confusion Matrix)로 "어떤 클래스를 자주 헷갈리는지" 봅니다.

In [ ]:
# ------------------------------------------------------------
# [9] 학습 곡선 + 테스트 평가 + 혼동행렬
# ------------------------------------------------------------
# (1) 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_history.png", dpi=140)
plt.show()

# (2) 테스트셋 평가
test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
print(f"테스트 정확도: {test_acc * 100:.2f}%  (loss={test_loss:.4f})")

y_prob = model.predict(X_test_s, verbose=0)
y_pred = np.argmax(y_prob, axis=1)  # 가장 확률 높은 클래스를 최종 예측으로

print("\nClassification Report")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

# (3) 혼동행렬
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (Test)")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=140)
plt.show()

## 10. 모델/스케일러 저장

나중에 노트북을 다시 열어도, 학습을 반복하지 않고 저장된 파일로 추론할 수 있습니다.

> 저장 위치는 `/content/outputs` 입니다. 런타임이 끊기면 사라지므로, 필요하면 좌측 파일 탐색기에서 내려받으세요.

In [ ]:
# ------------------------------------------------------------
# [10] 모델과 전처리 파라미터 저장
# ------------------------------------------------------------
MODEL_PATH = OUTPUT_DIR / "ito_cnn_model.keras"
SCALER_PATH = OUTPUT_DIR / "ito_scaler.npz"
META_PATH = OUTPUT_DIR / "model_meta.json"

model.save(MODEL_PATH)

# 추론 시에도 같은 표준화가 필요하므로 mean/scale을 함께 저장
np.savez(
    SCALER_PATH,
    mean=scaler.mean_.astype(np.float32),
    scale=scaler.scale_.astype(np.float32),
    freq=freq.astype(np.float32),
)

meta = {
    "class_names": CLASS_NAMES,
    "input_length": int(X.shape[1]),
    "test_accuracy": float(test_acc),
    "n_total": int(len(X)),
    "n_train": int(len(X_train)),
    "n_val": int(len(X_val)),
    "n_test": int(len(X_test)),
}
META_PATH.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

print("모델 저장   :", MODEL_PATH)
print("스케일러 저장:", SCALER_PATH)
print("메타정보    :", META_PATH)

## 11. 임의의 데이터 하나로 클래스 판정 (추론)

실무에서는 "새로운 측정 CSV 하나"를 넣고 고장 여부를 판단합니다.

아래에서는
1. 데이터셋에서 **랜덤 샘플**을 고르거나
2. `TARGET_FILE`에 **원하는 파일 경로**를 직접 지정

해 볼 수 있습니다.

In [ ]:
# ------------------------------------------------------------
# [11] 단일 샘플 추론
# ------------------------------------------------------------
def pick_random_sample(data_root: Path) -> Path:
    """7개 클래스 전체에서 CSV 하나를 무작위로 선택."""
    all_files = []
    for name in CLASS_NAMES:
        all_files.extend((data_root / name).glob("*.csv"))
    return random.choice(all_files)


def predict_one(csv_path: Path, model, mean, scale, class_names):
    """CSV 한 장을 읽어 전처리 후 클래스 확률을 계산."""
    _, meas, _ = load_csv_signal(csv_path)
    if len(meas) != len(mean):
        raise ValueError(f"신호 길이 불일치: {len(meas)} vs {len(mean)}")

    # 학습 때와 동일한 표준화 적용
    x = ((meas - mean) / scale).astype(np.float32)
    x = x[np.newaxis, :, np.newaxis]  # (1, L, 1)

    probs = model.predict(x, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    return pred_idx, class_names[pred_idx], probs, meas


# ----------------------------------------------------------
# ★ 수강생 실습: 여기만 바꿔서 원하는 파일을 판정해 보세요.
#   None  -> 랜덤 선택
#   문자열 -> DATA_ROOT 기준 상대경로 또는 절대경로
# 예) "M1/M1_S11_aug_0dB_0001.csv"
# 예) "Normal/Normal_S11_aug_0dB_0010.csv"
# ----------------------------------------------------------
TARGET_FILE = None

if TARGET_FILE is None:
    csv_path = pick_random_sample(DATA_ROOT)
else:
    csv_path = Path(TARGET_FILE)
    if not csv_path.is_absolute():
        csv_path = DATA_ROOT / csv_path

# 방금 학습한 model/scaler를 그대로 사용
# (노트북을 나중에 다시 열었다면 아래 두 줄의 주석을 해제하세요)
# model = tf.keras.models.load_model(OUTPUT_DIR / "ito_cnn_model.keras")
# _sc = np.load(OUTPUT_DIR / "ito_scaler.npz"); mean, scale = _sc["mean"], _sc["scale"]
mean = scaler.mean_.astype(np.float32)
scale = scaler.scale_.astype(np.float32)

pred_idx, pred_name, probs, meas = predict_one(
    csv_path, model, mean, scale, CLASS_NAMES
)

# 폴더명으로 실제 정답 추정 (강의 데이터셋 구조 기준)
true_name = None
for part in Path(csv_path).parts:
    if part in CLASS_NAMES:
        true_name = part
        break

print("=" * 52)
print("입력 파일 :", csv_path)
if true_name is not None:
    print("실제 클래스:", true_name)
print(f"예측 클래스: {pred_name}  (index={pred_idx})")
print("-" * 52)
print("클래스별 확률")
for i in np.argsort(probs)[::-1]:
    bar = "#" * int(probs[i] * 40)
    mark = "  <-- 예측" if i == pred_idx else ""
    print(f"  {CLASS_NAMES[i]:8s} {probs[i]*100:6.2f}%  {bar}{mark}")
print("=" * 52)
if true_name is not None:
    print("판정 결과:", "맞음 ✅" if true_name == pred_name else "틀림 ❌")

# 입력 파형도 함께 확인
plt.figure(figsize=(10, 4))
plt.plot(freq / 1e9, meas, linewidth=1.6)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Measurement (dB)")
title = f"Inference: pred={pred_name}"
if true_name is not None:
    title += f" | true={true_name}"
plt.title(title)
plt.grid(True, alpha=0.3)
plt.show()

## 12. (선택) 여러 파일을 연속으로 판정해보기

클래스마다 1개씩 뽑아 빠르게 맞는지 확인해 보는 셀입니다.

In [ ]:
# ------------------------------------------------------------
# [12] 클래스별 샘플 1개씩 추론 (데모)
# ------------------------------------------------------------
print(f"{'파일':40s} {'정답':8s} {'예측':8s} {'확률':>8s} 결과")
print("-" * 80)
for class_name in CLASS_NAMES:
    files = sorted((DATA_ROOT / class_name).glob("*.csv"))
    # 중간쯤 파일을 골라 너무 쉬운 첫 파일만 보지 않도록 함
    demo_path = files[len(files) // 2]
    pred_idx, pred_name, probs, _ = predict_one(
        demo_path, model, mean, scale, CLASS_NAMES
    )
    ok = "O" if pred_name == class_name else "X"
    print(
        f"{demo_path.name:40s} {class_name:8s} {pred_name:8s} "
        f"{probs[pred_idx]*100:7.2f}%   {ok}"
    )

## 정리 / 생각해볼 점

1. **입력은 이미지인가, 신호인가?** → 주파수 응답 곡선(1D)입니다.
2. **라벨은 어떻게 만들었나?** → 폴더 이름(`M1`, `Normal`, …)이 곧 정답입니다.
3. **왜 표준화가 필요한가?** → 스케일이 다른 입력을 안정적으로 학습하기 위함입니다.
4. **왜 Test를 남겨 두나?** → 본 적 없는 데이터에서의 일반화 성능을 공정하게 보기 위함입니다.
5. **실무 확장** → 노이즈 레벨이 다른 데이터, S21/S11 동시 입력, 더 깊은 CNN/Transformer 등을 시도할 수 있습니다.

---
**수고하셨습니다!**  
`TARGET_FILE`을 바꿔가며 여러 샘플을 판정해 보고, 혼동행렬에서 자주 틀리는 클래스 쌍을 관찰해 보세요.